# Phi-Instruct + SentenceTransformer

In [ ]:
# CONFIGURATION

TEST_PATH = "/kaggle/input/map-charting-student-math-misunderstandings/test.csv"
TRAIN_PATH = "/kaggle/input/map-charting-student-math-misunderstandings/train.csv"
SUBMISSION_PATH = "submission.csv"

X_COL_LABELS = ['row_id', 'QuestionText', 'MC_Answer', 'StudentExplanation']
Y_COL_LABELS = ['Category:Misconception', 'y_correct_response', 'y_misconception_exist', 'y_misconception_type']
COL_LABELS = X_COL_LABELS + Y_COL_LABELS
# misconception_categories = list(train_data.Misconception.dropna().unique())

CORRECT_LABELS = [
    'True_Correct', # Correct Answer with Correct Understanding
    'True_Neither', # Correct Answer with Unidentifiable Misconception
    'True_Misconception' # Correct Answer with Identifiable Misconception
]

INCORRECT_LABELS = [
    'False_Misconception',
    'False_Neither',
    'False_Correct'
]


CATEGORY_LABELS = CORRECT_LABELS + INCORRECT_LABELS

CATEGORY_META_PROMPT = {
    "True_Correct": (
        "The student’s response is correct (i.e. they do not have a misconception). "
        "It’s 'True' meaning the ground-truth answer is valid. No conceptual error."
    ),
    "True_Neither": (
        "The student’s response is not incorrect in a way that maps to a known misconception, "
        "but also not exactly 'fully correct'. This may capture ambiguous or edge cases "
        "where the explanation doesn’t reveal correctness or a misconception clearly."
    ),
    "True_Misconception": (
        "The student’s answer is technically correct, but their explanation reveals a misconception. "
        "They may have arrived at the right answer by flawed reasoning or a partial misunderstanding."
    ),
    "False_Misconception": (
        "The student’s response is incorrect and the explanation indicates a specific misconception. "
        "This represents a clear case of misunderstanding tied to a known misconception category."
    ),
    "False_Neither": (
        "The student’s response is incorrect, but the explanation does not map to a known misconception. "
        "Their reasoning is wrong, but not in a way that matches the predefined misconception labels."
    ),
    "False_Correct": (
        "The student’s response is incorrect, but their explanation shows correct reasoning. "
        "This likely reflects a minor slip or arithmetic mistake rather than a conceptual error."
    )
}

import json
import numpy as np
import pandas as pd
from pathlib import Path

## Data Loaders

In [ ]:
def setup_data_config(df: pd.DataFrame, label_path: str = "labels.json") -> list:
    if Path(label_path).exists():
        try:
            with open(label_path) as f:
                cat = json.load(f)
        except json.JSONDecodeError:
            # file exists but is empty or invalid JSON
            cat = df['Category:Misconception'].unique().tolist()
            with open(label_path, "w") as f:
                json.dump(cat, f, indent=2)
    else:
        cat = df['Category:Misconception'].unique().tolist()
        with open(label_path, "w") as f:
            json.dump(cat, f, indent=2)

    return cat

def load_train_dataset() -> pd.DataFrame:

    train_data = pd.read_csv(TRAIN_PATH)
    train_data.fillna('NA', inplace=True)

    train_data['Category:Misconception'] = train_data['Category'] + ":" + train_data['Misconception']

    train_data['y_correct_response'] = train_data['Category'].str.split('_').str[0]
    train_data['y_misconception_exist'] = train_data['Category'].str.split('_').str[-1]
    train_data['y_misconception_type'] = train_data['Misconception']

    cat_file = setup_data_config(train_data)
    return train_data[COL_LABELS], cat_file

train_data, CAT_ALL_LABELS = load_train_dataset()

## Data classes 

Data handlers to persist and handle data

In [ ]:
import random

def view_student_response(data: pd.DataFrame) -> None:
    # Pick a random question
    question = random.choice(data['QuestionText'].unique())
    print(f"Exam Question:\n{question}\n")

    # Group by MC_Answer for that question
    grp = data[data["QuestionText"] == question].groupby("MC_Answer")

    # Loop over each group
    for idx, df in grp:
        print(f"Student's Response: {idx}")
        print("Stats:\n-----")
        print("Exam Marked:")
        print(df["Category"].value_counts().to_markdown())
        print("\nMisconception:")
        print(df["Misconception"].value_counts().to_markdown())
        print("\n")

In [ ]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class PredLabel:
    _correct: bool = False
    _misconception_exist: Literal["Neither", "Correct", "Misconception"] | None = None
    _misconception_type: str | None = None
    pred_labels: list[str] = field(default_factory=lambda: {cat: 0 for cat in CAT_ALL_LABELS})

    @property
    def correct(self):
        return "True" if self._correct is True else "False"

    @correct.setter
    def correct(self, correct_answer: bool):
        self._correct = correct_answer
        label = "True" if correct_answer is True else "False"

        self.pred_labels = {key: val + 1 if label in key else val for key, val in self.pred_labels.items()}

    @property
    def misconception_exist(self):
        return self._misconception_exist

    @misconception_exist.setter
    def misconception_exist(self, misconception: Literal["Neither", "Correct", "Misconception"]):
        # Neither means not in misconception labels trained
        # Correct means explanation shows no misconception
        self._misconception_exist = misconception
        # label = self.correct + "_" + misconception
        correct_label = self.correct
        # all_cats = {"Neither", "Correct", "Misconception"}
        # miscon_remaining = all_cats - {misconception}

        pred_labels = self.pred_labels.copy()
        other_labels = [i for i in ['Neither', 'Correct', 'Misconception'] if i != misconception]

        for key, val in pred_labels.items():
            if correct_label in key and misconception not in key:
                if key in other_labels:
                    self.pred_labels[key] -= 0.5
                else:
                    self.pred_labels[key] -= 1
            elif correct_label in key and misconception in key:
                self.pred_labels[key] += 0.5

    @property
    def misconception_type(self):
        return self._misconception_type

    @misconception_type.setter
    def misconception_type(self, label: str):
        self._misconception_type = label
        label = self.correct + "_" + self.misconception_exist + ":" + label
        val = self.pred_labels.get(label, 0.0)

        self.pred_labels.update({
            label: val + 1
        })

    def to_submission_label(self):
        if isinstance(self.misconception_exist, str) and self.misconception_exist == "Misconception":
            raise ValueError("Cannot run final prediction labels. Misconception must be configured if misconception identified.")

        # Step 1: sort all labels by score
        sorted_items = sorted(self.pred_labels.items(), key=lambda x: x[1], reverse=True)
        return " ".join(sorted_items[:3])


In [ ]:
# DEMO TEST
pred = PredLabel()
# pred MC_Answer is correct
pred.correct = True
# pred Misconception exists
pred.misconception_exist = "Correct"
# pred Misconception type if exists
# pred.misconception_type = "NA"
pred.to_submission_label()

## Model I: Predict Incorrect Answer

Determines whether the selected answer is correct. (True or False in Category; e.g., True_Correct)

*Model Configuration (& Corresponding Column)*

**Input**

- Question (QuestionText)
- Student's Response (MC_Answer)
- Student's Explaination (StudentExplanation)

**Output**

- Binary Label Classifier (Category):
  - True
  - False

  Returns Prefix: `True_*/False_*`


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

# Point directly to the Kaggle dataset path (local snapshot you added as input)
INSTRUCT_MODEL_PATH = "/kaggle/input/phi-3/pytorch/phi-3.5-mini-instruct/2"
INSTRUCT_MODEL_ID = "microsoft/Phi-3-small-8k-instruct"

# Load tokenizer and model from local files
tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL_PATH,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True
)

In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer

SENTENCE_MODEL_ID = "all-MiniLM-L6-v2"
SENTENCE_MODEL_PATH = f"/root/.cache/huggingface/hub/models--sentence-transformers--{SENTENCE_MODEL_ID}"
SENTENCE_MODEL_REPO_ID = "sentence-transformers/all-MiniLM-L6-v2"
# DEFAULT : "/root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf"

def setup_sentence_encoder(repo_id: str = SENTENCE_MODEL_REPO_ID, local_path: str = SENTENCE_MODEL_PATH) -> SentenceTransformer:
    model_dir_path = Path(local_path) / "snapshots"

    if model_dir_path.exists():
        model_path = str(list(model_dir_path.iterdir())[0])
    else:
        from huggingface_hub import snapshot_download
        model_path = snapshot_download(repo_id=repo_id, repo_type="model")

    model = SentenceTransformer(model_path, local_files_only=True)
    model.training = False
    print(f'Configured Pooler Model Default Similarity Function to: {model.similarity_fn_name}')

    return model

pooler = setup_sentence_encoder()

In [ ]:
#from sentence_transformers import SentenceTransformer
#SENTENCE_MODEL_ID = "/root/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf"
#"sentence-transformers/all-MiniLM-L6-v2"
#pooler = SentenceTransformer(SENTENCE_MODEL_ID, local_files_only=True)
#pooler.training = False

In [ ]:
import numpy as np
from collections import namedtuple

from dataclasses import dataclass, field


Obs = namedtuple('Obs', ['correct', 'incorrect'])

HIDDEN_DIM = 384

FACT_INPUT_LABELS = [
    "QuestionId",
    "QuestionText"
    "CorrectResponse",
    "CorrectExplanation",
    "IncorrectResponse",
]

FACT_OUTPUT_LABELS = ['CorrectResponseVec', 'ContrastPair']  #'CorrectExplanationVec',

# observed input features
OBS_INPUT_LABELS = [
    "row_id"
    "QuestionId",
    "MC_Answer"
]

# labels that are returned from fn methods during runtime
OBS_OUTPUT_LABELS = ['PredFact', 'PredObs']

@dataclass
class ObserveHistory:
    correct: list = field(default_factory=list)
    incorrect: list = field(default_factory=list)

    @property
    def to_numpy(self):
        return (np.vstack(self.correct), np.vstack(self.incorrect))

@dataclass
class FactTable:
    # data input
    qid: int
    question_text: str
    # prompt output
    correct_response: str
    correct_explanation: str
    incorrect_response: str
    incorrect_explanation: str
    # sentence encoder output
    correct_response_vec: np.ndarray
    contrast_pair: np.ndarray
    observed: ObserveHistory = field(default_factory=ObserveHistory)

@dataclass
class Observation:
    # data input
    row_id: int
    qid: int
    question_text: str
    mc_answer: str
    student_explanation: str

class Gatekeeper:
    # tracks fact table, observations, positive/negative embeddings

    def __init__(
        self,
        model: SentenceTransformer,
        threshold: float = 0.5,
        obs_threshold: float = 0.5
    ):
        self.model = model

        # agent's fact table
        self.fact_table = {}
        # to avoid different rep of correct answer
        # self.obs_insights = {}

        self.threshold = threshold
        self.obs_threshold = obs_threshold

        self.input_fact_labels = FACT_INPUT_LABELS
        self.output_fact_labels = FACT_OUTPUT_LABELS
        self.fact_labels = self.input_fact_labels + self.output_fact_labels
        self.input_obs_labels = OBS_INPUT_LABELS
        self.output_obs_labels = OBS_OUTPUT_LABELS
        self.obs_labels = self.input_obs_labels + self.output_obs_labels

    def _pred_with_fact_table(self, qid: str, mc_answer: np.ndarray) -> dict | None:

        if fact := self.fact_table.get(qid, None):
            pos_neg = fact.contrast_pair

            if isinstance(pos_neg, np.ndarray):

                pred = self.model.similarity_pairwise(mc_answer, pos_neg)
                assert len(pred.shape) == 1 and pred.shape[0] == pos_neg.shape[0], f'pred_neg shape: {pos_neg.shape} and length of pred shape: {len(pred.shape)}'

                proba = pred / pred.sum()
                return dict(zip(('correct', 'incorrect'), proba))
            else:
                raise ValueError(f'Unexpected type saved for embedding: {pos_neg}')

        return None

    def _pred_with_observed(self, qid: str, mc_answer: np.ndarray):
        if fact_table := self.fact_table.get(qid, None):

            insights = fact_table.observed
            insights_correct, insights_incorrect = insights.to_numpy
            mean_pos_prob = mean_neg_prob = None

            if insights_correct.shape[0] > 0:
                pos_pred = self.model.similarity_pairwise(mc_answer, insights_correct)
                assert len(pos_pred.shape) == 1 and pos_pred.shape[0] == insights_correct.shape[0]
                mean_pos_prob = pos_pred.min()

                print('Pos prob mu: ', pos_pred.mean())
                print('Pos prob max: ', pos_pred.max())
                print('Pos prob min: ', pos_pred.min())

            if insights_incorrect.shape[0] > 0:
                neg_pred = self.model.similarity_pairwise(mc_answer, insights_incorrect)
                assert len(neg_pred.shape) == 1 and neg_pred.shape[0] == insights_incorrect.shape[0]
                mean_neg_prob = neg_pred.min()

                print('Neg prob mu: ', neg_pred.mean())
                print('Neg prob max: ', neg_pred.max())
                print('Neg prob min: ', neg_pred.min())

            if mean_pos_prob and mean_neg_prob:
                proba = np.array([mean_pos_prob.numpy(), mean_neg_prob.numpy()])
                prob = proba / proba.sum()

                if prob.argmax() == 0:
                    self.fact_table[qid].observed.correct.append(mc_answer)
                else:
                    self.fact_table[qid].observed.incorrect.append(mc_answer)

                return dict(zip(('correct', 'incorrect'), prob))

        print(f'Returning no pred from observations. Either: \n- Cannot find QuestionId {qid} in fact_table or obs_insights. Add to fact_table or obs_insights to initiate. OR\n- Error while normalizing probas.')
        return None

    def predict(self, qid: str, mc_answer: str | np.ndarray):
        assert qid in self.fact_table, f'Cannot find QuestionId {qid} in fact_table or obs_insights. Add to fact_table or obs_insights to initiate.'

        mc_answer_vec = self.model.encode([mc_answer]) if isinstance(mc_answer, str) else mc_answer

        z0 = self._pred_with_fact_table(qid, mc_answer_vec)
        z1 = self._pred_with_observed(qid, mc_answer_vec)
        print(f"Output Fact Table Pred: {z0}\nOutput Obs Pred: {z1}\n")

        return {
            "pred_fact_prob": z0,
            "pred_obs_prob": z1
        }

    def add_fact(self, **kwargs):
        assert set(kwargs.keys()) != set(self.input_fact_labels), f'Expected all of features in fact table to be defined, only have: {set(kwargs.keys()) - set(self.input_fact_labels)}'

        qid = kwargs.get('QuestionId', None)
        correct_answer = kwargs.get('CorrectResponse', None)
        correct_explanation = kwargs.get('CorrectExplanation', None)
        incorrect_answer = kwargs.get('IncorrectResponse', None)
        incorrect_explanation = kwargs.get('IncorrectExplanation', None)

        assert isinstance(correct_answer, str) and isinstance(correct_explanation, str)

        correct_answer_v = self.model.encode([f"Answer: {correct_answer}\n" + f"Explanation: {correct_explanation}"])
        incorrect_answer_v = self.model.encode([f"Answer: {incorrect_answer}\n" + f"Explanation: {incorrect_explanation}"])
        assert tuple(correct_answer_v.shape) == tuple((1, HIDDEN_DIM)) and tuple(incorrect_answer_v.shape) == tuple((1, HIDDEN_DIM))

        pro_neg = np.vstack([correct_answer_v, incorrect_answer_v])
        assert tuple(pro_neg.shape) == tuple((2, HIDDEN_DIM))

        #kwargs['CorrectResponseVec'] = correct_answer_v
        #kwargs['CorrectExplanationVec'] = correct_explanation_v
        #kwargs['ContrastPair'] = pro_neg

        # initialise
        # self.obs_insights[qid] = Obs(correct=correct_answer_v, incorrect=incorrect_answer_v)
        self.fact_table[qid] = FactTable(
            qid=qid,
            question_text=kwargs.get('QuestionText'),
            correct_response=correct_answer,
            correct_explanation=correct_explanation,
            incorrect_response=incorrect_answer,
            incorrect_explanation=incorrect_explanation,
            correct_response_vec=correct_answer_v,
            contrast_pair=pro_neg,
            observed=ObserveHistory(correct=[correct_answer_v], incorrect=[incorrect_answer_v])

        )

gate = Gatekeeper(model=pooler)

sample_fact = {
    "QuestionId": "Q1",
    "QuestionText": 'What fraction of the shape is not shaded? Give your answer in its simplest form. [Image: A triangle split into 9 equal smaller triangles. 6 of them are shaded.]',
    "CorrectResponse": "\\( \\frac{1}{3} \\)",
    "CorrectExplanation": "There are 9 equal parts in total. 6 are shaded and 3 are not shaded. The fraction not shaded is \\( \\tfrac{3}{9} \\), which simplifies to \\( \\tfrac{1}{3} \\).",
    "IncorrectResponse":  "\\( \\frac{6}{9} \\)",
    "IncorrectExplanation": "There are 9 equal parts in total. 6 are shaded and 3 are not shaded."
    # "IncorrectExplanation": "There are 6 shaded parts out of 9 in total. So the fraction not shaded is \\( \\tfrac{6}{9} \\), which simplifies to \\( \\tfrac{2}{3} \\). "
}

gate.add_fact(**sample_fact)
# True_Correct, False_Misconception/Neither, False_Correct
sample_output = [gate.predict(qid="Q1", mc_answer="\\( \\frac{1}{3} \\) There are 9 equal parts in total. 6 are shaded and 3 are not shaded."), gate.predict(qid="Q1", mc_answer="\\( \\frac{2}{3} \\). There are 6 shaded parts out of 9 in total. So the fraction not shaded is \\( \\tfrac{6}{9} \\), which simplifies to \\( \\tfrac{2}{3} \\)."), gate.predict(qid="Q1", mc_answer="4/5. There are 9 equal parts in total. 6 are shaded and 3 are not shaded. The fraction not shaded is \\( \\tfrac{3}{9} \\)")]
print('Obs shape:', len(gate.fact_table['Q1'].observed.correct), 'and incorrect:', len(gate.fact_table['Q1'].observed.incorrect))

## Data Preparation & Preprocessing

In [ ]:
# PREPARATION PROMPT
SYSTEM_PROMPT = """
You are a helpful assistant. When writing mathematical expressions, always format them as double-escaped LaTeX strings suitable for embedding inside Python or JSON.

- Wrap inline math in '\\(' and '\\)'.
- Escape every backslash with another backslash.
- Example: Instead of writing "1/3", output '\\( \\frac{1}{3} \\)'.
- Always return expressions in this form, e.g. '\\( x^2 + y^2 = z^2 \\)'.

Do not render math in plain text or Markdown, only in escaped LaTeX string form.
"""

ANTI_SYSTEM_PROMPT = """
You are an assistant that always responds with INCORRECT mathematical answers.
All answers must be formatted as double-escaped LaTeX strings suitable for Python or JSON.

Formatting rules:
- Wrap all math in '\\(' and '\\)'.
- Escape every backslash with another backslash.
- Example: Instead of giving the correct fraction 1/3, you might output '\\( \\frac{2}{3} \\)' or '\\( 5 \\)'.
- Do not explain or justify why it is wrong — simply provide the incorrect expression.
- Do not output plain text or Markdown. Always return only one escaped LaTeX string as the answer.
"""

# MISCONCEPTION EXISTENCE PROMPTS
STUDENT_RESPONSE_NARRATOR = """
You are an assistant summarizing the student's explanation for their selected response in a maths exam. Your task is to clean up the response and reformat in a third-person point of view that highlights the student's misconception within their response.
"""

MODEL_CONFIG = dict(
    # setup config
    expert_system_prompt=SYSTEM_PROMPT,
    anti_system_prompt=ANTI_SYSTEM_PROMPT,
    device="cuda",
    # tokenizer config
    add_generation_prompt=True,
    # generation config
    max_new_tokens=512,
)

class Basement:
    def __init__(
        self,
        model,
        tokenizer,
        **kwargs
    ):
        self.model = model
        self.tokenizer = tokenizer

        if kwargs:
            print(f"Updating model configuration: {kwargs}")
            MODEL_CONFIG.update(**kwargs)

        for key, val in MODEL_CONFIG.items():
            setattr(self, f"_{key}", val)

    def format_message(self, system_prompt: str, user_input: str, return_ids: bool = True) -> list[dict] | str:
        msg = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input}
        ]

        text = self.tokenizer.apply_chat_template(
            msg,
            tokenize=False,
            add_generation_prompt=self._add_generation_prompt
        )

        return text if return_ids is False else self.tokenizer([text], return_tensors="pt").to(self._device)

    def _generate_text(
        self,
        system_prompt: str,
        user_input: str,
    ):
        print(f'Passing system prompt: {system_prompt}')
        model_inputs = self.format_message(
            system_prompt=system_prompt,
            user_input=user_input
        )

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=self._max_new_tokens
        )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    def correct_response(self, question: str):
        return self._generate_text(
            system_prompt=self._expert_system_prompt,
            user_input=question
        )

    def incorrect_response(self, question: str):
        return self._generate_text(
            system_prompt=self._anti_system_prompt,
            user_input=question
        )


basement = Basement(model=model, tokenizer=tokenizer)

In [ ]:
correct_answer = basement.correct_response(question=example_question)
incorrect_answer = basement.incorrect_response(question=example_question)
print(f"Correct Response: {correct_answer}")
print(f"\n\nIncorrect Response: {incorrect_answer}")

example_student_answer = "\( \frac{3}{9} \)"

## Model II: Whether Incorrect Response Caused by Misconception

Assesses whether the explanation contains a misconception. (Correct, Misconception, or Neither in Category; e.g., True_Correct)

*Model Configuration (& Corresponding Column)*

**Input**

- Question (QuestionText)
- Student's Response (MC_Answer)
- Student's Explaination (StudentExplanation)
- Student's incorrect answer flagger (Category - from Model I)

**Output**

- Multi-label Classifier (Category):
  - '*_Correct'
  - '*_Neither'
  - '*_Misconception'
  Returns Suffix e.g. `True_*` or `False_*`, i.e. `[Correct, Neither, Misconception]`

## Model III: Type of Misconception Identified

Identifies the specific misconception present, i.e. whenever `False_*` and `*_Misconception` returned True from Model I and II, respectively.

*Model Configuration (& Corresponding Column)*

**Input**

- Question (QuestionText)
- Student's Response (MC_Answer)
- Student's Explaination (StudentExplanation)
- Student's incorrect answer flagger (Category - from Model I) i.e. only if `False_*` is captured.
- Student's Misconception Flagger (Category - from Model II) i.e. only if `*_Misonception` is captured / returned true.

**Output**

- Multi-labels (Misconception)
  Returns the type of misconception and suffixes `False_Misconception` etc.
  

## Wrap Up: Final Output

Entire Model Configuration 

**Input**

- Question (QuestionText)
- Student's Response (MC_Answer)
- Student's Explaination (StudentExplanation)

**Output**

- True_Correct: NA / *
- False_Neither: NA / *
- False_Misconception: NA / *

Output Example: 
```text
True_Correct:NA False_Neither:NA False_Misconception:Incomplete
```

In [ ]:
# main function to run in submission
Y_COL_LABEL = "Category:Misconception"
Y_COL_IDX = "row_id"

def prediction(question: str, mc_answer: str, student_explanation: str) -> str:
    #TODO: Run all models here. Output must be of type string.
    return "True_Correct:NA False_Neither:NA False_Misconception:Incomplete"

def main(test_path: str = TEST_PATH) -> None:
    preds = []
    df = pd.read_csv(test_path)

    for row_id, inputs in df.set_index('row_id').iterrows():
        preds += [
            {
                'row_id': row_id,
                Y_COL_LABEL: prediction(
                    question=inputs.get('QuestionText', ''),
                    mc_answer=inputs.get('MC_Answer', ''),
                    student_explanation=inputs.get('StudentExplanation', '')
                )
            }
        ]

    preds_df = pd.DataFrame(preds)
    print(f"{'*' * 10} Prediction Complete! {'*' * 10}")
    print(preds_df.to_markdown())

    preds_df.to_csv('submission.csv', index=False)
    print(f"{'*' * 10} Submission Complete! {'*' * 10}")

main()